# MLflow smoke test

Тестирование подключения к MLFLow из Jupyter Notebook

## Configuration

Edit the next cell, then run all cells top to bottom.

In [1]:
# Хак, чтобы добавить корень проекта в path для импорта модулей
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))
sys.path

['/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python312.zip',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor/.venv/lib/python3.12/site-packages',
 '/home/fiberfox/Projects/HSEAIMag2025']

In [2]:
# Код для загрузки конфига и подключения к MLFlow.

import os
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Literal

import mlflow
from dotenv import load_dotenv


def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (pyproject.toml). Open this notebook from the repo.")

def setup(environment: Literal["local", "prod"], experiment: str | None = None):
    CONFIG_BY_ENV = {
        "local": "config_local.toml",
        "prod": "config.toml",
    }

    if environment not in CONFIG_BY_ENV:
        raise ValueError(f"ENV must be one of {list(CONFIG_BY_ENV)}; got {environment!r}")

    REPO_ROOT = find_repo_root()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)

    load_dotenv(REPO_ROOT / ".env")

    os.environ["APP_CONFIG"] = CONFIG_BY_ENV[environment]

    from app.config.settings import get_settings

    get_settings.cache_clear()

    settings = get_settings().mlflow
    
    from app.mlflow import configure_mlflow
    configure_mlflow(experiment)

    resolved_experiment = experiment or settings.default_experiment
    print(f"Environment: {environment}")
    print(f"APP_CONFIG: {os.environ['APP_CONFIG']}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"S3 endpoint: {settings.s3_endpoint_url}")
    print(f"Experiment: {resolved_experiment}")

In [3]:
setup(environment='prod', experiment='test')

Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: test


## MLFlow

Пример логгирования эксперимента в MLFlow

In [4]:
run_name = f"smoke-test-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("source", "mlflow_smoke_test")
    mlflow.log_metric("ping", 1.0)
    mlflow.log_dict({"status": "ok"}, "smoke.json")

print(f"Run ID: {run.info.run_id}")
print(f"Run name: {run_name}")
print("Smoke test passed.")

🏃 View run smoke-test-20260530-093131 at: http://localhost:5050/#/experiments/2/runs/d073a2d084ce4d56b8e88ffb64b209ac
🧪 View experiment at: http://localhost:5050/#/experiments/2
Run ID: d073a2d084ce4d56b8e88ffb64b209ac
Run name: smoke-test-20260530-093131
Smoke test passed.


## Database queries

Примеры загрузки данных из PostgreSQL:

- Стоимость акций
- Данные по новостям: тикеры, сектор, sentiment

In [5]:
# Загрузка свечей MOEx

from app.core.database import AssetCandleRepository, get_db_session

TICKER = 'SBER'
LIMIT = 100

async with get_db_session() as session:
    repo = AssetCandleRepository(session)
    stocks_df = await repo.get_dataframe_by_ticker(TICKER, limit=LIMIT)

print(f'{TICKER}: {len(stocks_df)} rows')
stocks_df.tail()

SBER: 100 rows


,begin,open,high,low,close,volume,value,ticker
95,2026-05-23 14:00:00,322.79,322.80,322.60,322.75,98703.0,31854933.66,SBER
96,2026-05-23 15:00:00,322.75,322.80,322.60,322.64,65450.0,21122415.55,SBER
97,2026-05-23 16:00:00,322.64,322.79,322.57,322.65,29537.0,9531350.86,SBER
98,2026-05-23 17:00:00,322.65,322.75,322.51,322.65,64055.0,20665351.05,SBER
99,2026-05-23 18:00:00,322.64,322.77,322.60,322.72,52276.0,16868518.97,SBER


In [6]:
# Загрузка данных по новостям

from app.core.database import NewsArticleRepository, get_db_session

LIMIT = 100

async with get_db_session() as session:
    repo = NewsArticleRepository(session)
    enrichments_df = await repo.get_all_enrichments_as_dataframe(limit=LIMIT, ticker=TICKER, sector='MOEXFN')

print(f'Enrichments: {len(enrichments_df)} rows')
enrichments_df.head()

Enrichments: 100 rows


,id,news_article_id,published_at,tickers,sector,sentiment,sentiment_score
0,20676,188756,2025-10-24 10:08:00,"[YDEX, SBER, ROSN, VTBR, GAZP, LKOH]",MOEXFN,positive,0.965830
1,20731,188721,2025-10-24 14:29:00,[SBER],MOEXFN,positive,0.904252
2,20784,188684,2025-10-24 19:01:00,"[YDEX, SBER, ROSN, VTBR, TATN, GAZP, LKOH]",MOEXFN,negative,0.895174
3,20991,189693,2025-10-27 07:12:00,"[SBER, ROSN, LKOH, TATN]",MOEXFN,negative,0.547704
4,21010,189678,2025-10-27 10:10:00,"[YDEX, SBER, ROSN, VTBR, TATN, GAZP, LKOH]",MOEXFN,positive,0.646645
